# IndicPrayog-0.5B Custom 64K Tokenizer Pipeline

**What this notebook does (in order):**
1. **Sample** a balanced corpus from HuggingFace via *streaming* — no full downloads
2. **Train** a 64K byte-level BPE tokenizer on it
3. **Validate** fertility (tokens/word) per language against pass/fail targets
4. **Save** in HuggingFace format + play with it


**Data sources (verified live on HF API):**

| Language | Dataset | Config/Split | Byte share |
|:---------|:--------|:-------------|:----------:|
| Hindi    | `ai4bharat/sangraha` | `verified` / `hin` | 40% (600 MB) |
| English  | `HuggingFaceFW/fineweb-edu` | `sample-10BT` | 35% (525 MB) |
| Marathi  | `ai4bharat/sangraha` | `verified` / `mar` | 25% (375 MB) |

> **Why 40/35/25 and not the training mix 45/35/20?** The tokenizer must not starve Marathi —
> if Hindi dominates the merges, Marathi words fragment into more tokens forever. We upweight
> Marathi here, only for the tokenizer.

In [ ]:
# ── Cell 1: Install dependencies (run once) ──────────────────────────────
# datasets  : streaming download from HuggingFace
# tokenizers: HuggingFace's fast Rust BPE trainer
# transformers: to wrap the tokenizer in HF format at the end
%pip install -q datasets tokenizers transformers tqdm

In [ ]:
# ── Cell 2: Configuration — every knob in one place ─────────────────────
import os

# Folder layout (relative to this notebook, inside IndicPrayog_tokenizer/):
#   dataset/tokenizer_dataset/  ← sampled corpus lands here (~1.5 GB, delete after)
#   tokenizer/                  ← trained tokenizer lands here (keep forever)
CORPUS_DIR    = os.path.join("dataset", "tokenizer_dataset")
TOKENIZER_DIR = "tokenizer"
os.makedirs(CORPUS_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

# Balanced sampling plan — byte targets, NOT doc counts.
# (Marathi docs are shorter + Devanagari chars are 3 UTF-8 bytes each,
#  so counting docs would silently unbalance the corpus.)
MB = 1024 * 1024
SAMPLING_PLAN = [
    # (lang, dataset_id,                  config,        split,   target_bytes)
    ("hi", "ai4bharat/sangraha",          "verified",    "hin",   600 * MB),  # 40%
    ("en", "HuggingFaceFW/fineweb-edu",   "sample-10BT", "train", 525 * MB),  # 35%
    ("mr", "ai4bharat/sangraha",          "verified",    "mar",   375 * MB),  # 25%
]

VOCAB_SIZE  = 64_000   # matches model config vocab_size
MIN_DOC_LEN = 150      # skip junk docs shorter than this (chars)

# Special tokens — IDs fixed by order of appearance in trainer
SPECIAL_TOKENS = ["<pad>", "<eos>", "<bos>", "<unk>"]   # ids 0,1,2,3

print("Plan:")
for lang, ds, cfg, split, b in SAMPLING_PLAN:
    print(f"  {lang}: {ds} [{cfg}/{split}]  →  {b/MB:.0f} MB")

## Step 1 — Sample the corpus (streaming)

`streaming=True` means we read docs one at a time over HTTP and stop the moment we hit the
byte target — we never download the full dataset. Sangraha-verified-hin is only ~2.3 GB but
FineWeb-Edu is ~27 GB; streaming keeps both cheap.

Each doc is lightly cleaned (newlines → spaces) and written as one line into
`corpus/<lang>.txt`. Expect **~10–25 min total** depending on network speed.

In [ ]:
# ── Cell 3: Stream + sample each language to its byte target ────────────
from datasets import load_dataset
from tqdm.auto import tqdm

def sample_language(lang, dataset_id, config, split, target_bytes):
    """Stream docs from HF until we collect `target_bytes` of clean text."""
    out_path = os.path.join(CORPUS_DIR, f"{lang}.txt")

    # Resume-friendly: skip if already complete
    if os.path.exists(out_path) and os.path.getsize(out_path) >= target_bytes * 0.98:
        print(f"[{lang}] already sampled ({os.path.getsize(out_path)/MB:.0f} MB) — skipping")
        return

    print(f"[{lang}] streaming {dataset_id} ({config}/{split})...")
    ds = load_dataset(dataset_id, config, split=split, streaming=True)

    written, n_docs = 0, 0
    pbar = tqdm(total=target_bytes // MB, unit="MB", desc=f"sample-{lang}")
    with open(out_path, "w", encoding="utf-8") as f:
        for row in ds:
            text = row["text"].replace("\n", " ").strip()
            if len(text) < MIN_DOC_LEN:      # skip junk/navigation fragments
                continue
            line = text + "\n"
            nbytes = len(line.encode("utf-8"))
            f.write(line)
            written += nbytes
            n_docs += 1
            if written // MB > pbar.n:
                pbar.update(written // MB - pbar.n)
            if written >= target_bytes:
                break
    pbar.close()
    print(f"[{lang}] done: {n_docs:,} docs, {written/MB:.0f} MB → {out_path}\n")

for lang, ds_id, cfg, split, target in SAMPLING_PLAN:
    sample_language(lang, ds_id, cfg, split, target)

print("All languages sampled.")
total = sum(os.path.getsize(os.path.join(CORPUS_DIR, f)) for f in os.listdir(CORPUS_DIR))
print(f"Total corpus size: {total/MB:.0f} MB")

## Step 2 — Train the 64K byte-level BPE tokenizer

Design choices (and *why*):

| Choice | Why |
|:-------|:----|
| **NFC normalization** | Devanagari matras (े ि ो) have multiple Unicode encodings — NFC collapses them to one canonical form so the same word always tokenizes the same way |
| **Byte-level pre-tokenizer** | Any Unicode char is representable as bytes → `<unk>` almost never fires |
| **Split digits individually** | `2024` → `2 0 2 4` — models handle numbers far better this way |
| **Trains on all 3 files together** | BPE merges are frequency-based; the 40/35/25 byte balance controls whose subwords win |

Takes **~15–40 min on CPU** for 1.5 GB. Fully deterministic — same corpus → same tokenizer.

In [ ]:
# ── Cell 4: Train BPE ────────────────────────────────────────────────────
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel, Digits, Sequence
from tokenizers.normalizers import NFC
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
import time

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.normalizer = NFC()                       # canonical Devanagari
# NOTE: add_prefix_space=False. With True, ByteLevel injects a spurious leading-space
# marker at EVERY Digits split boundary (not just string-start), corrupting round-trip
# fidelity on any text containing digits and inflating fertility. Verified:
# pre_tokenize_str("पास4महीने") -> ('Ġ4', ...) with True vs ('4', ...) (correct) with False.
tokenizer.pre_tokenizer = Sequence([
    Digits(individual_digits=True),                 # 2024 → 2 0 2 4
    ByteLevel(add_prefix_space=False),              # byte fallback for any char
])
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,                                # a merge must appear ≥2× to exist
    special_tokens=SPECIAL_TOKENS,                  # <pad>=0 <eos>=1 <bos>=2 <unk>=3
    initial_alphabet=ByteLevel.alphabet(),          # guarantee all 256 bytes present
    show_progress=True,
)

files = [os.path.join(CORPUS_DIR, f"{lang}.txt") for lang, *_ in SAMPLING_PLAN]
print("Training on:", files)
t0 = time.time()
tokenizer.train(files, trainer=trainer)
print(f"\nTrained in {(time.time()-t0)/60:.1f} min — vocab size: {tokenizer.get_vocab_size():,}")

raw_path = os.path.join(TOKENIZER_DIR, "tokenizer.json")
tokenizer.save(raw_path)
print(f"Saved → {raw_path}")

## Step 3 — Validate: fertility (tokens per word)

**Fertility = tokens ÷ words.** Lower is better — it means the tokenizer compresses that
language efficiently. This is THE gate before packing 12B tokens.

| Language | Target | Hard reject |
|:---------|:------:|:-----------:|
| Hindi    | ~1.8   | > 2.5 |
| Marathi  | ~2.0   | > 2.8 |
| English  | ~1.3   | > 1.8 |

We measure on **held-out text** (fresh docs streamed past the training sample), not on
the training corpus itself — same principle as any ML eval.

In [ ]:
# ── Cell 5: Fertility on held-out samples ────────────────────────────────
from tokenizers import Tokenizer

tok = Tokenizer.from_file(os.path.join(TOKENIZER_DIR, "tokenizer.json"))

# Quick sanity sentences first (instant feedback)
quick_tests = [
    ("hi", "मैं आज बाजार जा रहा हूँ और वहाँ से ताज़ी सब्ज़ियाँ खरीदूंगा।"),
    ("mr", "मी आज बाजारात जात आहे आणि तेथून ताज्या भाज्या विकत घेईन."),
    ("en", "The quick brown fox jumps over the lazy dog near the river bank."),
]
print("Quick sanity check:")
for lang, text in quick_tests:
    ids = tok.encode(text).ids
    f = len(ids) / len(text.split())
    print(f"  {lang}: {len(ids):3d} tokens / {len(text.split()):2d} words = {f:.2f} t/w")

# Round-trip check — decode(encode(x)) must return x
sample = quick_tests[0][1]
assert tok.decode(tok.encode(sample).ids).strip() == sample, "Round-trip FAILED!"
print("\nRound-trip (encode→decode) OK ✓")

In [ ]:
# ── Cell 6: Proper fertility eval on held-out streamed docs ─────────────
from datasets import load_dataset

TARGETS = {"hi": (1.8, 2.5), "mr": (2.0, 2.8), "en": (1.3, 1.8)}  # (target, hard_reject)
HELDOUT_DOCS = 2_000   # docs per language for the eval

results = {}
for lang, ds_id, cfg, split, _ in SAMPLING_PLAN:
    ds = load_dataset(ds_id, cfg, split=split, streaming=True)
    # skip past roughly what training sampled, then take fresh docs
    it = iter(ds.skip(300_000))
    n_tokens, n_words, taken = 0, 0, 0
    for row in it:
        text = row["text"].replace("\n", " ").strip()
        if len(text) < MIN_DOC_LEN:
            continue
        n_tokens += len(tok.encode(text).ids)
        n_words  += len(text.split())
        taken += 1
        if taken >= HELDOUT_DOCS:
            break
    fert = n_tokens / max(n_words, 1)
    target, reject = TARGETS[lang]
    status = "✓ PASS" if fert <= reject else "✗ FAIL — rebalance corpus & retrain!"
    results[lang] = fert
    print(f"{lang}: fertility = {fert:.2f} t/w   (target ~{target}, reject >{reject})   {status}")

assert all(results[l] <= TARGETS[l][1] for l in results), \
    "Fertility gate FAILED — do NOT proceed to data packing. Rebalance SAMPLING_PLAN and retrain."
print("\nAll languages passed the fertility gate ✓ — safe to pack 12B tokens with this tokenizer.")

## Step 4 — Save in HuggingFace format

Wraps the raw `tokenizer.json` in `PreTrainedTokenizerFast` so `AutoTokenizer.from_pretrained()`
works everywhere (training loop, eval scripts, inference).

In [ ]:
# ── Cell 7: HF wrapper + save ────────────────────────────────────────────
from transformers import PreTrainedTokenizerFast

hf_tok = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(TOKENIZER_DIR, "tokenizer.json"),
    pad_token="<pad>",
    eos_token="<eos>",
    bos_token="<bos>",
    unk_token="<unk>",
    model_max_length=131072,   # matches model max_position_embeddings
)
hf_tok.save_pretrained(TOKENIZER_DIR)

# verify special token IDs are exactly what the model config expects
for name in SPECIAL_TOKENS:
    print(f"  {name} → id {hf_tok.convert_tokens_to_ids(name)}")
print("\nSaved HF tokenizer →", TOKENIZER_DIR)
print(os.listdir(TOKENIZER_DIR))

## Step 5 — Playground: see how it tokenizes

Run any text through it and inspect the pieces. Useful for building intuition about what
the model will actually "see".

In [ ]:
# ── Cell 8: Inspect tokenization of any text ────────────────────────────
def show(text):
    enc = tok.encode(text)
    pieces = [tok.decode([i]) for i in enc.ids]
    print(f"text : {text}")
    print(f"ids  : {enc.ids}")
    print(f"parts: {' | '.join(repr(p) for p in pieces)}")
    print(f"count: {len(enc.ids)} tokens for {len(text.split())} words\n")

show("भारत एक विशाल देश है।")                    # Hindi
show("महाराष्ट्रातील पुणे हे शिक्षणाचे माहेरघर आहे.")  # Marathi
show("Machine learning models need good tokenizers.")  # English
show("मैंने ChatGPT से पूछा कि weather कैसा है")       # Hinglish code-switching

## Step 6 — (Optional) Upload to HF + clean local corpus

Uncomment and run when happy with the fertility numbers. The 1.5 GB `corpus/` folder is
safe to delete after — the tokenizer is all that matters going forward.

In [ ]:
# ── Cell 9: Upload tokenizer to HF dataset repo, then free disk ─────────
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=TOKENIZER_DIR,
#     repo_id="YOUR_HF_USERNAME/IndicPrayog-0.5B-dataset",
#     repo_type="dataset",
#     path_in_repo="tokenizer",
# )
# print("Tokenizer uploaded.")

# import shutil
# shutil.rmtree(CORPUS_DIR)     # free the ~1.5 GB corpus
# print("corpus/ deleted — tokenizer/ kept.")